# PG Data Service — Enriched Pipeline Validation

Walk through the enriched pipeline output and validate against the Domo Ad Performance card CSV.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.max_columns', 35)
pd.set_option('display.width', 200)

## 1. Enriched Data (internal column names)

In [ ]:
from api import get_enriched

enriched = get_enriched("2026-03-15", "2026-03-18")
print(f"{len(enriched)} ads, {len(enriched.columns)} columns")
enriched.head(3)

In [ ]:
# All available columns
print("Columns:")
for c in enriched.columns:
    print(f"  {c}")

## 2. Ad Performance Card (Domo display names)

In [ ]:
from scripts.export_ad_performance import get_ad_performance_card

card = get_ad_performance_card("2026-03-15", "2026-03-18")
print(f"{len(card)} rows, {card['Ad'].nunique()} ads, {card['Day'].nunique()} days, {len(card.columns)} columns")
card.head(5)

In [ ]:
# Top 10 by spend (aggregate across days for ranking)
daily_spend = card.groupby("Ad")["Spend"].sum().nlargest(10).index
card[card["Ad"].isin(daily_spend)][["Ad", "Day", "Spend", "Net Revenue", "Net ROAS", "Gross Revenue", "Orders", "# SC Trials<BR>Started", "NC Orders"]]

## 3. Compare Against Domo CSV

Export a fresh CSV from the Domo Ad Performance card and update the path below.

In [ ]:
# UPDATE THIS PATH to your fresh Domo CSV export
DOMO_CSV = "~/Downloads/Ad Performance.csv"

domo = pd.read_csv(DOMO_CSV)
domo["Ad"] = domo["Ad"].astype(str).str.lower().str.strip()
print(f"Domo CSV: {len(domo)} rows, {domo['Day'].nunique()} days, {domo['Ad'].nunique()} unique ads")

In [ ]:
# Aggregate both to per-ad totals for comparison
additive = ["Spend", "Gross Revenue", "Clicks", "Impressions", "Orders",
            "# SC Trials<BR>Started", "NC Orders", "Net Revenue", "Fixed Refund<BR>Net Revenue"]

for c in additive:
    if c in domo.columns:
        domo[c] = pd.to_numeric(domo[c], errors="coerce").fillna(0)

domo_agg = domo.groupby("Ad", as_index=False)[additive].sum()

# Aggregate our daily rows too
ours = card.copy()
ours["Ad"] = ours["Ad"].astype(str).str.lower().str.strip()
for c in additive:
    if c in ours.columns:
        ours[c] = pd.to_numeric(ours[c], errors="coerce").fillna(0)
ours_agg = ours.groupby("Ad", as_index=False)[additive].sum()

print(f"Domo: {len(domo_agg)} ads")
print(f"Ours: {len(ours_agg)} ads")

In [ ]:
# Find common ads
common = set(domo_agg["Ad"]) & set(ours_agg["Ad"])
print(f"Common ads: {len(common)}")
print(f"Domo-only: {len(set(domo_agg['Ad']) - set(ours_agg['Ad']))}")
print(f"Ours-only: {len(set(ours_agg['Ad']) - set(domo_agg['Ad']))}")

In [ ]:
# Side-by-side: top 10 by spend
d = domo_agg[domo_agg["Ad"].isin(common)].set_index("Ad")
o = ours_agg[ours_agg["Ad"].isin(common)].set_index("Ad")

top_ads = d.nlargest(10, "Spend").index

for col in additive:
    domo_vals = d.loc[top_ads, col]
    ours_vals = pd.to_numeric(o.reindex(top_ads)[col], errors="coerce")
    diff = (ours_vals - domo_vals).abs()
    rel = diff / domo_vals.abs().clip(lower=0.01)
    status = "OK" if rel.max() <= 0.01 else "MISMATCH"
    print(f"  {status:10s} | {col:35s} | max diff: {rel.max():.4%}")

In [ ]:
# Detailed side-by-side for top 5 ads
for ad in top_ads[:5]:
    print(f"\n{'='*80}")
    print(f"Ad: {ad[:70]}")
    print(f"{'='*80}")
    for col in additive:
        d_val = d.loc[ad, col]
        o_val = pd.to_numeric(o.loc[ad, col], errors="coerce") if ad in o.index else float("nan")
        match = "ok" if abs(d_val - o_val) / max(abs(d_val), 0.01) < 0.01 else "!!"
        print(f"  {col:35s}  domo={d_val:>15.2f}  ours={o_val:>15.2f}  {match}")

## 4. Quick Tess Classification Preview

In [ ]:
# Classification using Tess PRD v1.4 thresholds
def classify(row):
    if row["spend"] < 2500:
        return "Testing"
    elif row["net_roas"] >= 1.0:
        return "Winner"
    elif row["net_roas"] >= 0.80:
        return "Potential"
    else:
        return "Underperformer"

enriched["classification"] = enriched.apply(classify, axis=1)
print(enriched["classification"].value_counts())
print(f"\nTotal spend: ${enriched['spend'].sum():,.2f}")
print(f"Total revenue: ${enriched['gross_revenue'].sum():,.2f}")

In [ ]:
# Winners
winners = enriched[enriched["classification"] == "Winner"].nlargest(10, "spend")
winners[["ad_name", "funnel", "spend", "gross_revenue", "net_roas", "nc_net_roas", "cpa", "sc_trials"]]

In [ ]:
# Underperformers (spend >= $2500 but net_roas < 0.80)
under = enriched[enriched["classification"] == "Underperformer"].nlargest(10, "spend")
under[["ad_name", "funnel", "spend", "gross_revenue", "net_roas", "nc_net_roas", "cpa", "sc_trials"]]

In [ ]:
# Breakdown by funnel
by_funnel = enriched.groupby("funnel").agg(
    ads=("ad_name", "count"),
    total_spend=("spend", "sum"),
    total_revenue=("gross_revenue", "sum"),
    winners=("classification", lambda x: (x == "Winner").sum()),
).sort_values("total_spend", ascending=False)

by_funnel["roas"] = by_funnel["total_revenue"] / by_funnel["total_spend"]
by_funnel